# Notebook 02 — Process

Aligns raw rasters, computes distances, builds LQ surfaces, runs SLX causal test.
Requires outputs from `01_download.ipynb`.

In [ ]:
# papermill parameters — overridden at runtime
config_path = "configs/template.yaml"
runs_root = "runs"
run_dir = None   # if set, loads an existing run instead of creating a new one

In [ ]:
from palmoil_risk.io.run import create_run, load_run

if run_dir:
    ctx = load_run(run_dir)
else:
    ctx = create_run(config_path, runs_root=runs_root)
print(f"Run dir: {ctx.run_dir}")

In [ ]:
from palmoil_risk.data.user_inputs import ingest_user_inputs

inputs = ingest_user_inputs(ctx)
print("User inputs available:", {k: str(v) for k, v in inputs.items() if v})

In [ ]:
from palmoil_risk.process.align import align_all, compute_all_distances

aligned = align_all(ctx, inputs)
print("Aligned rasters:", list(aligned.keys()))

In [ ]:
dist_paths = compute_all_distances(ctx)
print("Distance rasters:", list(dist_paths.keys()))

In [ ]:
from palmoil_risk.process.lq import compute_lq, classify_lq_zones, compute_mill_kde

m_raster = ctx.data_dir / "M.tif"
p_raster = ctx.data_dir / "P.tif"
lq_mp_out = ctx.data_dir / "lq_mp.tif"
lq_pm_out = ctx.data_dir / "lq_pm.tif"

compute_lq(m_raster, p_raster, lq_mp_out, lq_pm_out, epsilon=ctx.config.lq_epsilon)
print("LQ surfaces computed")

lq_col = "lq_mp" if ctx.config.lq_direction == "mp" else "lq_pm"
classify_lq_zones(ctx.data_dir / f"{lq_col}.tif", ctx.data_dir / "lq_sq.tif")
print("LQ zones classified")

In [ ]:
mill_gpkg = ctx.raw_dir / "mill" / "mill.gpkg"
if mill_gpkg.exists():
    kde_path = ctx.data_dir / "mill_kde.tif"
    compute_mill_kde(
        mill_gpkg, ctx.data_dir / "forest_t2.tif",
        bandwidth_km=ctx.config.kde_bandwidth_km,
        output_path=kde_path,
    )
    print(f"Mill KDE written to {kde_path}")
else:
    print("No mill.gpkg found — KDE skipped")

In [ ]:
from palmoil_risk.process.correlation import run_correlation_pipeline

corr_results = run_correlation_pipeline(ctx)
direction = corr_results.get("slx", {}).get("direction", "N/A")
print(f"Causal direction: {direction}")